# Entrega 5 - Comparação Definitiva de Arquiteturas
Neste notebook, consolidamos os aprendizados da nossa etapa de busca em grade (Grid Search) e expandimos as avaliações para comparar diretamente o melhor modelo CNN encontrado contra diversas variações de Vision Transformers.

## Arquitetura CNN Utilizada: EfficientNetB0
Para o modelo convolucional base, utilizamos a **EfficientNetB0**.
* **Como ela funciona:** É uma arquitetura otimizada que aplica o conceito de *Compound Scaling* (Escalonamento Composto). Em vez de aumentar arbitrariamente a profundidade (número de camadas), a largura (número de canais) ou a resolução da imagem de entrada separadamente, a EfficientNet os escala de maneira uniforme e matemática, conseguindo maior acurácia com um número de parâmetros muito menor que a concorrência (como ResNet ou VGG).
* **Camadas e Blocos:** A EfficientNetB0 possui **237 camadas** no total, estruturadas principalmente ao redor de **7 blocos principais** chamados de MBConv (*Mobile Inverted Bottleneck Convolution*). Estes blocos utilizam *Depthwise Separable Convolutions* para reduzir drasticamente a carga computacional, mantendo o poder de extração de padrões locais (bordas, texturas, formas típicas de lesões de pele).

Neste notebook, carregaremos nosso melhor modelo CNN (com Learning Rate 1e-05 e Dropout 0.5) e treinaremos 5 modelos Transformers diferentes com **exatamente os mesmos parâmetros** de treinamento.


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import pandas as pd
import seaborn as sns
import tf_keras as keras
import tensorflow as tf
from tf_keras.models import Model
from tf_keras.layers import Dense, GlobalAveragePooling2D, Dropout, Concatenate, Input, StringLookup, Embedding, Flatten
from tf_keras.applications import EfficientNetB0
from transformers import TFAutoModelForImageClassification, TFAutoModel
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# --- HIPERPARÂMETROS EXATOS DA MELHOR CNN ---
# Reproduzindo as mesmas condições da Grid Search da Entrega 3
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-05
PATIENCE = 6

# --- CONFIGURAÇÕES DE DIRETÓRIO ---
PASTAS_DATASET = ['imagens_acrais_benignas', 'imagens_acrais_maligno']
PASTA_ISIC = 'ISIC-images'
MODELO_CNN_PATH = 'Modelos/treino_Modelos/LR1e-05_DO0.5.keras'
TAMANHO_IMAGEM = (224, 224)

# Configurar GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"Erro GPU: {e}")


In [ ]:
print("Carregando Dados...")
dfs = []
for pasta in PASTAS_DATASET:
    caminho_csv = os.path.join(pasta, 'metadata.csv')
    if os.path.exists(caminho_csv):
        df_temp = pd.read_csv(caminho_csv)
        df_temp['caminho_imagem'] = df_temp['isic_id'].apply(lambda x: os.path.join(pasta, f"{x}.jpg"))
        dfs.append(df_temp)

df_completo = pd.concat(dfs, ignore_index=True)
df_existente = df_completo[df_completo['caminho_imagem'].apply(os.path.exists)].copy()
df_existente['anatom_site_general'] = df_existente['anatom_site_general'].fillna('unknown')

df_benigno = df_existente[df_existente['diagnosis_1'] == 'Benign']
df_maligno = df_existente[df_existente['diagnosis_1'] == 'Malignant']
df_filtrado = pd.concat([df_benigno, df_maligno]).sample(frac=1, random_state=42).reset_index(drop=True)
df_filtrado['label'] = (df_filtrado['diagnosis_1'] == 'Malignant').astype(int)

# ISIC Extra Dataset
df_isic = pd.read_csv(os.path.join(PASTA_ISIC, 'metadata.csv'))
df_isic['caminho_imagem'] = df_isic['isic_id'].apply(lambda x: os.path.join(PASTA_ISIC, f"{x}.jpg"))
df_isic = df_isic.drop_duplicates(subset='isic_id')
df_isic = df_isic[df_isic['caminho_imagem'].apply(os.path.exists)].copy()
df_isic_filtrado = df_isic[df_isic['diagnosis_1'].isin(['Benign', 'Malignant'])].copy().reset_index(drop=True)
df_isic_filtrado['label'] = (df_isic_filtrado['diagnosis_1'] == 'Malignant').astype(int)
df_isic_filtrado['anatom_site_general'] = df_isic_filtrado['anatom_site_general'].fillna('unknown')


In [ ]:
# Divisão e Oversampling (igual às entregas anteriores)
df_train, df_val = train_test_split(df_filtrado, test_size=0.2, random_state=42, stratify=df_filtrado['label'])

df_malignant = df_train[df_train['label'] == 1]
df_benign = df_train[df_train['label'] == 0]
maior_classe = max(len(df_malignant), len(df_benign))

df_benign_up = resample(df_benign, replace=True, n_samples=maior_classe, random_state=42)
df_malignant_up = resample(df_malignant, replace=True, n_samples=maior_classe, random_state=42)
df_treino_equilibrado = pd.concat([df_malignant_up, df_benign_up]).sample(frac=1, random_state=42).reset_index(drop=True)

# Datasets TF.Data
def process_path(file_path, site, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, TAMANHO_IMAGEM)
    img = img / 255.0
    return {'imagem': img, 'site': site}, label

data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.125),
    keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
])

def augment(inputs, labels):
    img = data_augmentation(inputs['imagem'], training=True)
    return {'imagem': img, 'site': inputs['site']}, labels

# Train
train_ds = tf.data.Dataset.from_tensor_slices((df_treino_equilibrado['caminho_imagem'].values, df_treino_equilibrado['anatom_site_general'].values, df_treino_equilibrado['label'].values))
train_ds = train_ds.map(process_path).batch(BATCH_SIZE).map(augment).prefetch(tf.data.AUTOTUNE)

# Validação
val_ds = tf.data.Dataset.from_tensor_slices((df_val['caminho_imagem'].values, df_val['anatom_site_general'].values, df_val['label'].values))
val_ds = val_ds.map(process_path).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ISIC
isic_ds = tf.data.Dataset.from_tensor_slices((df_isic_filtrado['caminho_imagem'].values, df_isic_filtrado['anatom_site_general'].values, df_isic_filtrado['label'].values))
isic_ds = isic_ds.map(process_path).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Datasets simples (sem dict) para Swin, DeiT e ViT (que não usam multimodalidade)
train_ds_single = train_ds.map(lambda x, y: (x['imagem'], y))
val_ds_single = val_ds.map(lambda x, y: (x['imagem'], y))
isic_ds_single = isic_ds.map(lambda x, y: (x['imagem'], y))


In [ ]:
# 1. Carregando a Melhor CNN
print(f"Carregando Modelo CNN Salvo: {MODELO_CNN_PATH}")
modelo_cnn = tf.keras.models.load_model(MODELO_CNN_PATH)
modelo_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Funções construtoras dos Transformers

# Fix para o Swin Transformer
import transformers.models.swin.modeling_tf_swin as swin_module
from transformers.tf_utils import shape_list
def custom_window_reverse(windows: tf.Tensor, window_size: int, height: int, width: int) -> tf.Tensor:
    channels = shape_list(windows)[-1]
    windows = tf.reshape(windows, (-1, height // window_size, width // window_size, window_size, window_size, channels))
    windows = tf.transpose(windows, (0, 1, 3, 2, 4, 5))
    windows = tf.reshape(windows, (-1, height, width, channels))
    return windows
swin_module.window_reverse = custom_window_reverse

def compile_transformer(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), 
        loss='binary_crossentropy', 
        metrics=['accuracy']
    )
    return model

def build_swin_model():
    base = TFAutoModelForImageClassification.from_pretrained("microsoft/swin-base-patch4-window7-224", num_labels=1, ignore_mismatched_sizes=True, use_safetensors=False)
    if hasattr(base, "swin"): base.swin.trainable = False
    
    inputs = Input(shape=(224, 224, 3), name='imagem')
    outputs = base(pixel_values=tf.transpose(inputs, perm=[0, 3, 1, 2])).logits
    model = Model(inputs=inputs, outputs=keras.activations.sigmoid(outputs), name="Swin_Transformer")
    return compile_transformer(model)

def build_deit_model():
    base = TFAutoModelForImageClassification.from_pretrained("facebook/deit-base-patch16-224", num_labels=1, ignore_mismatched_sizes=True, use_safetensors=False)
    if hasattr(base, "deit"): base.deit.trainable = False
    
    inputs = Input(shape=(224, 224, 3), name='imagem')
    outputs = base(pixel_values=tf.transpose(inputs, perm=[0, 3, 1, 2])).logits
    model = Model(inputs=inputs, outputs=keras.activations.sigmoid(outputs), name="DeiT_III")
    return compile_transformer(model)

def build_vit_base_model():
    # NOVO MODELO (ViT Original do Google)
    base = TFAutoModelForImageClassification.from_pretrained("google/vit-base-patch16-224", num_labels=1, ignore_mismatched_sizes=True, use_safetensors=False)
    if hasattr(base, "vit"): base.vit.trainable = False
    
    inputs = Input(shape=(224, 224, 3), name='imagem')
    outputs = base(pixel_values=tf.transpose(inputs, perm=[0, 3, 1, 2])).logits
    model = Model(inputs=inputs, outputs=keras.activations.sigmoid(outputs), name="ViT_Base_Google")
    return compile_transformer(model)

def build_multimodal_vit():
    base_vit = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k", use_safetensors=False)
    base_vit.trainable = False
    
    img_input = Input(shape=(224, 224, 3), name='imagem')
    vit_features = base_vit(pixel_values=tf.transpose(img_input, perm=[0, 3, 1, 2])).pooler_output
    
    site_input = Input(shape=(1,), dtype=tf.string, name='site')
    vocab = np.unique(df_filtrado['anatom_site_general'].astype(str))
    lookup = StringLookup(vocabulary=vocab, output_mode='int')(site_input)
    site_features = Flatten()(Embedding(input_dim=len(vocab)+2, output_dim=16)(lookup))
    
    x = Dropout(0.3)(Dense(128, activation='relu')(Concatenate()([vit_features, site_features])))
    model = Model(inputs={'imagem': img_input, 'site': site_input}, outputs=Dense(1, activation='sigmoid')(x), name="Multimodal_ViT")
    return compile_transformer(model)

def build_mixed_cnn_vit():
    img_input = Input(shape=(224, 224, 3), name='imagem')
    
    base_cnn = EfficientNetB0(weights="imagenet", include_top=False, input_tensor=img_input)
    base_cnn.trainable = False
    for layer in base_cnn.layers:
        if layer.name.startswith("block7") or layer.name.startswith("top_"):
            if not isinstance(layer, keras.layers.BatchNormalization): layer.trainable = True
                
    cnn_features = Dropout(0.4)(GlobalAveragePooling2D()(base_cnn.output))
    
    base_vit = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k", use_safetensors=False)
    base_vit.trainable = False
    vit_features = Dropout(0.4)(base_vit(pixel_values=tf.transpose(img_input, perm=[0, 3, 1, 2])).pooler_output)
    
    x = Dropout(0.3)(Dense(256, activation='relu')(Concatenate()([cnn_features, vit_features])))
    model = Model(inputs=img_input, outputs=Dense(1, activation='sigmoid')(x), name="Mixed_CNN_ViT")
    return compile_transformer(model)


## Treinamento Simultâneo dos Transformers
Nesta célula, treinamos os modelos Transformers. **Importante:** Todos os parâmetros estão estritamente iguais à melhor CNN. Note que o `BATCH_SIZE = 32` pode causar exigências elevadas de memória VRAM dependendo do hardware.


In [ ]:
# Instanciando os modelos
modelos_transformers = {
    "Swin": (build_swin_model(), train_ds_single, val_ds_single),
    "DeiT": (build_deit_model(), train_ds_single, val_ds_single),
    "ViT_Original": (build_vit_base_model(), train_ds_single, val_ds_single),
    "ViT_Multimodal": (build_multimodal_vit(), train_ds, val_ds),
    "Mixed_CNN_ViT": (build_mixed_cnn_vit(), train_ds_single, val_ds_single)
}

# Callback idêntico ao da CNN
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=PATIENCE, 
    restore_best_weights=True, 
    verbose=1
)

historicos = {}

print("=== INICIANDO TREINAMENTO DOS TRANSFORMERS ===")
for nome, (modelo, t_ds, v_ds) in modelos_transformers.items():
    print(f"\nTreinando: {nome}...")
    hist = modelo.fit(
        t_ds,
        epochs=EPOCHS,
        validation_data=v_ds,
        callbacks=[early_stop]
    )
    historicos[nome] = hist

print("\nTreinamento Finalizado!")


## Comparação de Todos os Modelos (CNN + Transformers)
Abaixo agrupamos os modelos recém-treinados com o modelo CNN previamente treinado para uma comparação definitiva das matrizes de confusão.


In [ ]:
# Dicionário com todos os modelos treinados para avaliação final
todos_modelos = {
    "CNN (EfficientNetB0)": (modelo_cnn, val_ds_single, isic_ds_single)
}

# Adicionando os transformers na lista final
for nome, (modelo, t_ds, v_ds) in modelos_transformers.items():
    # Pega o dataset ISIC correspondente (single vs dict)
    i_ds = isic_ds if nome == "ViT_Multimodal" else isic_ds_single
    todos_modelos[nome] = (modelo, v_ds, i_ds)

resultados_finais = {}

for nome, (modelo, v_ds, i_ds) in todos_modelos.items():
    print(f"\nAvaliando: {nome}")
    
    # Avaliação em ISIC
    y_true_isic = []
    y_pred_probs_isic = []
    for batch_x, batch_y in i_ds:
        probs = modelo.predict(batch_x, verbose=0)
        y_pred_probs_isic.extend(probs.flatten())
        y_true_isic.extend(batch_y.numpy())
        
    y_pred_isic = (np.array(y_pred_probs_isic) > 0.5).astype(int)
    
    print(classification_report(y_true_isic, y_pred_isic, target_names=["Benign", "Malignant"]))
    
    # Salvamos para o plot
    resultados_finais[nome] = confusion_matrix(y_true_isic, y_pred_isic)

# --- Gráfico de Matrizes de Confusão ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (nome, cm) in enumerate(resultados_finais.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
    axes[i].set_title(f"{nome}\nISIC-images")
    axes[i].set_ylabel('Real')
    axes[i].set_xlabel('Predito')

# Limpando o eixo não utilizado (pois temos 6 modelos e uma grid 2x3 fechada perfeitamente)
plt.tight_layout()
plt.show()
